<a href="https://colab.research.google.com/github/kumakun2008/CIFAR-10_Comp/blob/main/kumakun2008_ResNet_1116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-10 画像分類（完全スクラッチ学習・GPU高速化版：ResNet-18 + WideResNet-28-10 + DenseNet-121）

## 工夫した点（箇条書き）

<table>
  <tr>
    <td><img src="https://github.com/kumakun2008/CIFAR-10_Comp/blob/main/assets/gemini_icon.png?raw=1" width="84" alt="生産者: Gemin 4 Hydrogen" /></td>
    <td>
      <strong>【生産者表示】Gemin 4 Hydrogen が作りました！！！</strong><br/>
      （顔写真付き・CIFAR-10高効率学習チューニング済み）
    </td>
  </tr>
</table>

* **生産者**：**Gemin 4 Hydrogenが作りました！！！**（上記の顔写真付き）
* **事前学習済みの重みは一切使用せず、CIFAR-10の訓練データのみからスクラッチ学習**を実施した。
* **32×32解像度に最適化したモデル構造へ変更**：
  * 配布コードのAlexNet（入り口のstride=2畳み込み＋MaxPoolで8×8に縮小・巨大FC層による過学習）から変更し、`torchvision` の `ResNet-18` と `DenseNet-121` の入力層を `3×3 conv (stride=1)`・プーリングなし（`nn.Identity()`）に改変して32×32の空間解像度を保持した。
  * さらにCIFAR-10スクラッチ学習で高い表現力を持つ **WideResNet-28-10**（Pre-activation残差ブロック＋Dropout）を実装し、組み合わせた。
* **学習手法と最適化の改善**：
  * 配布コードでエポックループ内にあった `optimizer` の再初期化をループ外へ修正。
  * 最適化手法を **SGD（lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4）** に変更し、**CosineAnnealingLR** で学習率を滑らかに減衰させた。
  * **Label Smoothing（0.05）** と **AMP（bfloat16 / float16 混合精度学習）** を導入した。
* **データ拡張とGPU一括高速化**：
  * CIFAR-10全画像をVRAM上に常駐させ、`RandomCrop(32, padding=4, reflect)`・`RandomHorizontalFlip(0.5)`・**Cutout**・**Mixup（alpha=0.2）** をすべてGPU上でバッチ一括処理することで、CPU DataLoaderボトルネックを解消し数倍〜10倍近い高速化を実現した。
* **検証（Validation）に基づく設計確定と仕上げ（テストデータは学習・選定に不使用）**：
  * まず訓練45,000枚 / 検証5,000枚に分割し、`Val. Acc`・検証データの混同行列・誤分類可視化で設定と学習曲線を確認した。
  * 設定確定後、訓練＋検証を合わせた **50,000枚全量** で3種類の異種モデル（`ResNet-18`・`WideResNet-28-10`・`DenseNet-121`）をそれぞれスクラッチで再学習した。
  * 最終テスト評価では、各モデルの単体精度に加え、**TTA（元画像＋左右反転画像の確率平均）と3モデルのアンサンブル**を適用した。


In [ ]:
import os
import copy
import time
import random
import shutil
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.datasets as datasets
from torchvision import models

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
SEED = 1234

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


# CIFAR-10 データの読み込みと前処理（GPU常駐高速化）

In [ ]:
ROOT = '.data'
os.makedirs(ROOT, exist_ok=True)
tar_path = os.path.join(ROOT, 'cifar-10-python.tar.gz')
drive_tar_path = '/content/drive/MyDrive/Colab_Data/CIFAR10/cifar-10-python.tar.gz'
mirror_url = 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'

if not os.path.exists(os.path.join(ROOT, 'cifar-10-batches-py')) and (
    not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071
):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        if os.path.exists(drive_tar_path):
            print(f'Copying CIFAR-10 from Google Drive: {drive_tar_path}')
            shutil.copy2(drive_tar_path, tar_path)
    except Exception as e:
        print(f'Google Drive copy skipped ({e})')

    if not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071:
        print(f'Downloading CIFAR-10 from mirror: {mirror_url}')
        urllib.request.urlretrieve(mirror_url, tar_path)

train_raw = datasets.CIFAR10(root=ROOT, train=True, download=True)
test_raw = datasets.CIFAR10(root=ROOT, train=False, download=True)

means = train_raw.data.mean(axis=(0, 1, 2)) / 255
stds = train_raw.data.std(axis=(0, 1, 2)) / 255

print(f'Calculated means: {means}')
print(f'Calculated stds: {stds}')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = (use_amp and amp_dtype == torch.float16)

mean_t = torch.tensor(means, dtype=torch.float32).view(1, 3, 1, 1)
std_t = torch.tensor(stds, dtype=torch.float32).view(1, 3, 1, 1)

X_all_cpu = torch.from_numpy(train_raw.data).permute(0, 3, 1, 2).float().div_(255.0)
X_all_cpu = (X_all_cpu - mean_t) / std_t
Y_all_cpu = torch.tensor(train_raw.targets, dtype=torch.long)

X_test_cpu = torch.from_numpy(test_raw.data).permute(0, 3, 1, 2).float().div_(255.0)
X_test_cpu = (X_test_cpu - mean_t) / std_t
Y_test_cpu = torch.tensor(test_raw.targets, dtype=torch.long)

VALID_RATIO = 0.9
n_train_examples = int(len(X_all_cpu) * VALID_RATIO)

split_generator = torch.Generator().manual_seed(SEED)
perm_idx = torch.randperm(len(X_all_cpu), generator=split_generator)
train_idx = perm_idx[:n_train_examples]
valid_idx = perm_idx[n_train_examples:]

X_full = X_all_cpu.to(device)
Y_full = Y_all_cpu.to(device)
X_train = X_full[train_idx]
Y_train = Y_full[train_idx]
X_valid = X_full[valid_idx]
Y_valid = Y_full[valid_idx]
X_test = X_test_cpu.to(device)
Y_test = Y_test_cpu.to(device)

classes = test_raw.classes
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 100

print(f'Device: {device} | AMP: {use_amp} ({amp_dtype})')
print(f'Train: {len(X_train)} | Valid: {len(X_valid)} | Full: {len(X_full)} | Test: {len(X_test)}')


In [ ]:
def normalize_image(image):
    image = image.clone()
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min=image_min, max=image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image

def plot_images(images, labels, classes, normalize=False):
    n_images = len(images)
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(10, 10))
    for i in range(rows * cols):
        ax = fig.add_subplot(rows, cols, i + 1)
        image = images[i]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')

N_IMAGES = 25
plot_images(X_train[:N_IMAGES], Y_train[:N_IMAGES].tolist(), classes, normalize=True)


# モデル設計（すべて事前学習なし・32×32 CIFAR-10向けスクラッチ構築）
1. **ResNet-18（CIFAR-10改変版）**
2. **WideResNet-28-10**
3. **DenseNet-121（CIFAR-10改変版）**


In [ ]:
def build_resnet18_cifar(num_classes=10):
    model = models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


class WideBasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropout_rate=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.dropout = nn.Dropout(p=dropout_rate) if dropout_rate > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)

        if stride != 1 or in_planes != out_planes:
            self.shortcut = nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(self.dropout(F.relu(self.bn2(out))))
        return out + self.shortcut(x)


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=10, dropout_rate=0.3, num_classes=10):
        super().__init__()
        assert (depth - 4) % 6 == 0, 'WideResNet depth should be 6n + 4'
        n = (depth - 4) // 6
        n_stages = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]

        self.conv1 = nn.Conv2d(3, n_stages[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._wide_layer(WideBasicBlock, n_stages[0], n_stages[1], n, dropout_rate, stride=1)
        self.layer2 = self._wide_layer(WideBasicBlock, n_stages[1], n_stages[2], n, dropout_rate, stride=2)
        self.layer3 = self._wide_layer(WideBasicBlock, n_stages[2], n_stages[3], n, dropout_rate, stride=2)
        self.bn1 = nn.BatchNorm2d(n_stages[3])
        self.linear = nn.Linear(n_stages[3], num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def _wide_layer(self, block, in_planes, out_planes, num_blocks, dropout_rate, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        curr_planes = in_planes
        for s in strides:
            layers.append(block(curr_planes, out_planes, s, dropout_rate))
            curr_planes = out_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def build_densenet121_cifar(num_classes=10):
    model = models.densenet121(weights=None, num_classes=num_classes)
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()
    return model


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


OUTPUT_DIM = 10
print(f'ResNet-18 (CIFAR-10) params:   {count_parameters(build_resnet18_cifar(OUTPUT_DIM)):,}')
print(f'WideResNet-28-10 params:       {count_parameters(WideResNet(28, 10, 0.3, OUTPUT_DIM)):,}')
print(f'DenseNet-121 (CIFAR-10) params:{count_parameters(build_densenet121_cifar(OUTPUT_DIM)):,}')


# GPU一括データ拡張・学習・評価関数（Mixup / Cutout / AMP / TTA対応）

In [ ]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc


def augment_cifar_batch(x, pad=4, erase_prob=0.5):
    B, C, H, W = x.shape
    x_pad = F.pad(x, (pad, pad, pad, pad), mode='reflect')
    top = torch.randint(0, 2 * pad + 1, (B, 1, 1, 1), device=x.device)
    left = torch.randint(0, 2 * pad + 1, (B, 1, 1, 1), device=x.device)
    rows = top + torch.arange(H, device=x.device).view(1, 1, H, 1)
    cols = left + torch.arange(W, device=x.device).view(1, 1, 1, W)
    b_idx = torch.arange(B, device=x.device).view(B, 1, 1, 1)
    c_idx = torch.arange(C, device=x.device).view(1, C, 1, 1)
    out = x_pad[b_idx, c_idx, rows, cols]

    flip_mask = (torch.rand(B, 1, 1, 1, device=x.device) < 0.5)
    out = torch.where(flip_mask, torch.flip(out, dims=[3]), out)

    if erase_prob > 0:
        cy = torch.randint(0, H, (B, 1, 1, 1), device=x.device)
        cx = torch.randint(0, W, (B, 1, 1, 1), device=x.device)
        half_h = torch.randint(3, 8, (B, 1, 1, 1), device=x.device)
        half_w = torch.randint(3, 8, (B, 1, 1, 1), device=x.device)
        apply_cut = (torch.rand(B, 1, 1, 1, device=x.device) < erase_prob)
        yy = torch.arange(H, device=x.device).view(1, 1, H, 1)
        xx = torch.arange(W, device=x.device).view(1, 1, 1, W)
        erase_mask = (
            apply_cut
            & (yy >= cy - half_h) & (yy < cy + half_h)
            & (xx >= cx - half_w) & (xx < cx + half_w)
        )
        out = out.masked_fill(erase_mask, 0.0)
    return out


def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


def train_epoch_gpu(model, X_data, Y_data, batch_size, optimizer, criterion, scaler,
                    mixup_alpha=0.2, use_amp=True, amp_dtype=torch.float16):
    model.train()
    n = X_data.size(0)
    perm = torch.randperm(n, device=X_data.device)
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        x = augment_cifar_batch(X_data[idx])
        y = Y_data[idx]

        optimizer.zero_grad(set_to_none=True)

        if mixup_alpha > 0:
            x_mixed, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x_mixed)
                loss = mixup_criterion(criterion, y_pred, y_a, y_b, lam)
            acc = lam * calculate_accuracy(y_pred, y_a) + (1.0 - lam) * calculate_accuracy(y_pred, y_b)
        else:
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_acc += acc.item()
        num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def evaluate_gpu(model, X_data, Y_data, batch_size, criterion, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    n = X_data.size(0)
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    with torch.no_grad():
        for start in range(0, n, batch_size):
            x = X_data[start:start + batch_size]
            y = Y_data[start:start + batch_size]
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)
            total_loss += loss.item()
            total_acc += acc.item()
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


# Phase 1：検証データ（Val. Acc）を用いたモデル・ハイパーパラメータ検証
まずは `ResNet-18（CIFAR-10改変版）` を使って、訓練45,000枚 / 検証5,000枚で学習曲線と `Val. Acc` を確認します（テストデータは一切参照しません）。


In [ ]:
EPOCHS = 100
LR = 0.1
WEIGHT_DECAY = 5e-4
MIXUP_ALPHA = 0.2
LABEL_SMOOTHING = 0.05

set_seed(SEED)
val_model = build_resnet18_cifar(OUTPUT_DIM).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
eval_criterion = nn.CrossEntropyLoss().to(device)

optimizer = optim.SGD(val_model.parameters(),
                      lr=LR,
                      momentum=0.9,
                      weight_decay=WEIGHT_DECAY,
                      nesterov=True)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

best_valid_acc = 0.0
best_epoch = 0
history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

for epoch in trange(EPOCHS, desc="Phase 1 (Val Check)"):
    start_time = time.monotonic()

    train_loss, train_acc = train_epoch_gpu(
        val_model, X_train, Y_train, BATCH_SIZE, optimizer, criterion, scaler,
        mixup_alpha=MIXUP_ALPHA, use_amp=use_amp, amp_dtype=amp_dtype
    )
    valid_loss, valid_acc = evaluate_gpu(
        val_model, X_valid, Y_valid, EVAL_BATCH_SIZE, eval_criterion,
        use_amp=use_amp, amp_dtype=amp_dtype
    )
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(val_model.state_dict(), 'resnet18_val_best.pt')

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch + 1) == EPOCHS:
        print(f'Epoch: {epoch+1:03} | Time: {epoch_mins}m {epoch_secs}s | LR: {scheduler.get_last_lr()[0]:.5f}')
        print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
        print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

print(f'\nBest Val. Acc: {best_valid_acc*100:.2f}% (at Epoch {best_epoch})')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['valid_loss'], label='Valid Loss')
axes[0].set_title('Loss Curve (Phase 1)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot([a * 100 for a in history['train_acc']], label='Train Acc')
axes[1].plot([a * 100 for a in history['valid_acc']], label='Valid Acc')
axes[1].set_title('Accuracy Curve (Phase 1)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
plt.show()


## 検証データ（Validation）での誤差分析（混同行列・誤分類画像）
テストデータを分析に使うのではなく、検証データで弱点を確認します。


In [ ]:
def get_predictions_gpu(model, X_data, Y_data, batch_size, use_tta=False, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, X_data.size(0), batch_size):
            x = X_data[start:start + batch_size]
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_prob = F.softmax(model(x).float(), dim=-1)
                if use_tta:
                    y_prob_flip = F.softmax(model(torch.flip(x, dims=[3])).float(), dim=-1)
                    y_prob = 0.5 * (y_prob + y_prob_flip)
            probs.append(y_prob.cpu())
    return X_data.cpu(), Y_data.cpu(), torch.cat(probs, dim=0)


def plot_confusion_matrix(labels, pred_labels, classes):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = confusion_matrix(labels, pred_labels)
    cm = ConfusionMatrixDisplay(cm, display_labels=classes)
    cm.plot(values_format='d', cmap='Blues', ax=ax)
    plt.xticks(rotation=20)


def plot_most_incorrect(incorrect, classes, n_images, normalize=True):
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(20, 16))
    for i in range(min(rows * cols, len(incorrect))):
        ax = fig.add_subplot(rows, cols, i + 1)
        image, true_label, probs = incorrect[i]
        image = image.permute(1, 2, 0)
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        true_class = classes[true_label]
        incorrect_class = classes[incorrect_label]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.cpu().numpy())
        ax.set_title(f'true: {true_class} ({true_prob:.3f})\npred: {incorrect_class} ({incorrect_prob:.3f})')
        ax.axis('off')
    fig.subplots_adjust(hspace=0.4)


val_model.load_state_dict(torch.load('resnet18_val_best.pt', map_location=device))
val_images, val_labels, val_probs = get_predictions_gpu(
    val_model, X_valid, Y_valid, EVAL_BATCH_SIZE, use_tta=True, use_amp=use_amp, amp_dtype=amp_dtype
)
val_preds = torch.argmax(val_probs, dim=1)
print(f'Validation Acc (with TTA): {(val_preds == val_labels).float().mean().item()*100:.2f}%')
plot_confusion_matrix(val_labels, val_preds, classes)


In [ ]:
corrects = torch.eq(val_labels, val_preds)
incorrect_examples = []
for image, label, prob, correct in zip(val_images, val_labels, val_probs, corrects):
    if not correct:
        incorrect_examples.append((image, label, prob))

incorrect_examples.sort(reverse=True, key=lambda x: torch.max(x[2], dim=0).values)
plot_most_incorrect(incorrect_examples, classes, 25)


# Phase 2：設定確定後、訓練＋検証の全50,000枚で3モデルを学習（アンサンブル用）
Phase 1 で確認した設定（`EPOCHS=100`, `SGD + CosineAnnealingLR`, `Cutout + Mixup + Label Smoothing`）を使い、訓練＋検証の **50,000枚すべて** で構造の異なる3モデルをゼロから学習します。
1. **ResNet-18（CIFAR-10改変版）**
2. **WideResNet-28-10**
3. **DenseNet-121（CIFAR-10改変版）**


In [ ]:
def train_full_dataset(model_fn, model_name, save_path, epochs=100, seed=1234):
    set_seed(seed)
    model = model_fn().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
    optimizer = optim.SGD(model.parameters(),
                          lr=LR,
                          momentum=0.9,
                          weight_decay=WEIGHT_DECAY,
                          nesterov=True)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

    print(f'=== Start Full Training: {model_name} ({count_parameters(model):,} params) ===')
    for epoch in trange(epochs, desc=model_name):
        start_time = time.monotonic()
        train_loss, train_acc = train_epoch_gpu(
            model, X_full, Y_full, BATCH_SIZE, optimizer, criterion, scaler,
            mixup_alpha=MIXUP_ALPHA, use_amp=use_amp, amp_dtype=amp_dtype
        )
        scheduler.step()
        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        if (epoch + 1) % 20 == 0 or epoch == 0 or (epoch + 1) == epochs:
            print(f'[{model_name}] Epoch {epoch+1:03}/{epochs} ({epoch_mins}m {epoch_secs}s) | '
                  f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')

    torch.save(model.state_dict(), save_path)
    print(f'Saved {model_name} weights to {save_path}\n')
    return model


In [ ]:
FULL_EPOCHS = 100

model_resnet18 = train_full_dataset(
    lambda: build_resnet18_cifar(OUTPUT_DIM),
    model_name='ResNet-18 (CIFAR-10)',
    save_path='resnet18_full.pt',
    epochs=FULL_EPOCHS,
    seed=SEED
)

model_wrn28_10 = train_full_dataset(
    lambda: WideResNet(depth=28, widen_factor=10, dropout_rate=0.3, num_classes=OUTPUT_DIM),
    model_name='WideResNet-28-10',
    save_path='wrn28_10_full.pt',
    epochs=FULL_EPOCHS,
    seed=SEED + 1
)

model_densenet121 = train_full_dataset(
    lambda: build_densenet121_cifar(OUTPUT_DIM),
    model_name='DenseNet-121 (CIFAR-10)',
    save_path='densenet121_full.pt',
    epochs=FULL_EPOCHS,
    seed=SEED + 2
)


# 最終テスト評価（10,000枚）

In [ ]:
# 各モデル単体のテスト精度（配布コード準拠の評価）
model_resnet18.load_state_dict(torch.load('resnet18_full.pt', map_location=device))
model_wrn28_10.load_state_dict(torch.load('wrn28_10_full.pt', map_location=device))
model_densenet121.load_state_dict(torch.load('densenet121_full.pt', map_location=device))

for name, m in [('ResNet-18', model_resnet18),
                ('WideResNet-28-10', model_wrn28_10),
                ('DenseNet-121', model_densenet121)]:
    test_loss, test_acc = evaluate_gpu(
        m, X_test, Y_test, EVAL_BATCH_SIZE, eval_criterion, use_amp=use_amp, amp_dtype=amp_dtype
    )
    print(f'[{name}] Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


In [ ]:
# 3モデル アンサンブル + TTA（左右反転平均）による最終テスト評価
def evaluate_ensemble_tta_gpu(models_list, X_data, Y_data, batch_size, use_tta=True,
                              use_amp=True, amp_dtype=torch.float16):
    for m in models_list:
        m.eval()

    n = X_data.size(0)
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    with torch.no_grad():
        for start in range(0, n, batch_size):
            x = X_data[start:start + batch_size]
            y = Y_data[start:start + batch_size]

            probs_sum = 0.0
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                for m in models_list:
                    probs = F.softmax(m(x).float(), dim=-1)
                    if use_tta:
                        probs_flip = F.softmax(m(torch.flip(x, dims=[3])).float(), dim=-1)
                        probs = 0.5 * (probs + probs_flip)
                    probs_sum = probs_sum + probs

            avg_probs = probs_sum / len(models_list)
            loss = F.nll_loss(torch.log(avg_probs.clamp(min=1e-12)), y)
            acc = calculate_accuracy(avg_probs, y)

            total_loss += loss.item()
            total_acc += acc.item()
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


ensemble_models = [model_resnet18, model_wrn28_10, model_densenet121]
test_loss, test_acc = evaluate_ensemble_tta_gpu(
    ensemble_models, X_test, Y_test, EVAL_BATCH_SIZE, use_tta=True, use_amp=use_amp, amp_dtype=amp_dtype
)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')
